# Bitcoin Classical Baselines

## Role
Artifact-only comparison of final past-only classical systems, with an explicitly historical static-protocol appendix.

## Inputs
Frozen validated forecasts, canonical target, and historical executed static-protocol RMSE values recovered from Git history.

## Outputs
Final classical metrics, seasonality evidence, and a labelled historical static-vs-fair comparison.

## Depends On
01 Bitcoin Data/EDA.

## Authoritative Status
AUTHORITATIVE ANALYSIS

## What This Notebook Does Not Do
It does not fit or overwrite classical model vectors. The appendix does not recreate historical forecasts or treat their static protocol as authoritative.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
_, target=load_bitcoin_target(ROOT); train,test=canonical_split(target); v=load_validated_forecasts(ROOT); f=forecast_series(v,target); names=['Naive','7-Day Moving Average','Simple Exponential Smoothing — Rolling One-Step','Additive-Trend Exponential Smoothing','ARIMA Rolling One-Step']; metric_table(v.Actual,{n:f[n] for n in names},train).sort_values('RMSE')

,MAE,RMSE,MAPE,sMAPE,MASE,N,Relative MAE vs Naive
Model,,,,,,,
Naive,1290.353242,1853.624774,1.742747,1.744142,4.575633,1061,1.000000
Simple Exponential Smoothing — Rolling One-Step,1290.358684,1855.731424,1.742685,1.743871,4.575652,1061,1.000004
ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209,4.609396,1061,1.007379
Additive-Trend Exponential Smoothing,1308.541314,1871.702185,1.763640,1.763424,4.640128,1061,1.014095
7-Day Moving Average,2209.776153,2999.605073,3.021810,3.024208,7.835935,1061,1.712536


## Seasonality decision
Lag-7 ACF = -0.023409; approximate 95% bound = ±0.026920; weekly STL strength = 0.069923. Weekly SARIMA was not retained.

## Protocol note
SES and additive-trend smoothing refit daily on 128 prior prices. ARIMA starts with 128 returns and appends each newly observed return without refitting parameters.

## Appendix: Historical Static-vs-Fair Protocol Comparison
The static RMSE values below are historical executed outputs recovered from the pre-rebuild `02_Classical_Models.ipynb` in Git history (`113de8e^`). They are retained only to show the effect of replacing an unrealistic fixed-origin multi-step protocol with the authoritative past-only rolling one-step protocol. No historical forecast vector is regenerated here. The apparent 25–28× failure largely disappears when each model uses only information genuinely available before each forecast date.

In [3]:
historical_static_rmse = pd.Series({
    'Simple Exponential Smoothing': 52421.328707,
    'Additive-Trend Exponential Smoothing': 48571.096828,
    'ARIMA': 52421.667992,
}, name='Historical Static-Protocol RMSE')
fair_name = {
    'Simple Exponential Smoothing': 'Simple Exponential Smoothing — Rolling One-Step',
    'Additive-Trend Exponential Smoothing': 'Additive-Trend Exponential Smoothing',
    'ARIMA': 'ARIMA Rolling One-Step',
}
fair_metrics = metric_table(v.Actual, {n: f[n] for n in fair_name.values()}, train)
fair_rmse = pd.Series({label: fair_metrics.loc[model, 'RMSE'] for label, model in fair_name.items()}, name='Fair Rolling One-Step RMSE')
static_vs_fair = pd.concat([historical_static_rmse, fair_rmse], axis=1)
static_vs_fair['Static / Fair RMSE Ratio'] = static_vs_fair['Historical Static-Protocol RMSE'] / static_vs_fair['Fair Rolling One-Step RMSE']
static_vs_fair

,Historical Static-Protocol RMSE,Fair Rolling One-Step RMSE,Static / Fair RMSE Ratio
Simple Exponential Smoothing,52421.328707,1855.731424,28.248338
Additive-Trend Exponential Smoothing,48571.096828,1871.702185,25.950227
ARIMA,52421.667992,1866.302859,28.088511
